# Controlled AI-Document Generation Feasibility Study (Colab GPU Launcher)

> **Tên đề tài chính thức**: “Nghiên cứu và xây dựng công cụ phát hiện ảnh và tài liệu giả mạo sinh bởi AI.”<br>
> **Protocol**: Version 1.2.0 (Amendment v1.2.0)<br>
> **Mục tiêu**: Thử nghiệm khả thi tạo dữ liệu trên đúng **bốn tài liệu CORD v2** (2 replacement, 2 removal) bằng `diffusers/stable-diffusion-xl-1.0-inpainting-0.1`.<br>
> **Ranh giới**: Đây là generation feasibility study, KHÔNG phải detector training, KHÔNG phải cohort đánh giá độc lập.<br>
> **Không yêu cầu token/đăng nhập**: Chạy hoàn toàn bằng tài khoản Google Colab miễn phí thông thường (GPU T4). Không yêu cầu Hugging Face token.

### Quy Trình 4 Bước Thực Hiện Trên Colab:
1. **Cell 1**: Kiểm tra phần cứng GPU và cài đặt thư viện cần thiết (`diffusers`, `transformers`, `accelerate`, `safetensors`, `pyarrow`, `pillow`).
2. **Cell 2**: Tái tạo tự động kế hoạch máy tiền đăng ký `document_feasibility_plan.json` và mã nguồn thực thi `run_document_feasibility_colab.py` (đối soát SHA-256 bit-exact).
3. **Cell 3**: Thực thi 4 attempts đã khóa cứng: tải checkpoint FP16 (6.46 GiB), nạp 4 ảnh CORD-v2 từ cache, inpainting suy luận, composite và kiểm tra bitwise equality ngoài mask.
4. **Cell 4**: Tải gói kết quả `document_feasibility_results.zip` về máy LOCAL để kiểm chứng kỹ thuật và thẩm định nội dung.


In [ ]:
# Cell 1: Kiểm tra phần cứng GPU và cài đặt dependencies
!nvidia-smi

print("Installing exact pinned dependencies...")
!pip install -q diffusers==0.32.2 transformers==4.49.0 accelerate==1.4.0 safetensors==0.5.3 pyarrow==19.0.1

import diffusers, transformers, accelerate, safetensors, pyarrow, PIL
print(f"[OK] diffusers: {diffusers.__version__}")
print(f"[OK] transformers: {transformers.__version__}")
print(f"[OK] accelerate: {accelerate.__version__}")
print(f"[OK] safetensors: {safetensors.__version__}")
print(f"[OK] pyarrow: {pyarrow.__version__}")
print(f"[OK] Pillow: {PIL.__version__}")

import torch
assert torch.cuda.is_available(), "LỖI: Chưa kích hoạt GPU! Vui lòng vào Runtime -> Change runtime type -> Chọn T4 GPU!"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[OK] GPU sẵn sàng: {gpu_name} ({vram_gb:.2f} GB VRAM)")
if vram_gb < 12.0:
    print(f"CẢNH BÁO: VRAM ({vram_gb:.2f} GB) thấp hơn 12 GB. Attention slicing sẽ được kích hoạt để tránh OOM.")


In [ ]:
# Cell 2: Staging kế hoạch máy tiền đăng ký và runner script
import json
import hashlib
import sys
from pathlib import Path

if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

EXPECTED_PLAN_SHA = "4320eb1e85149323afeed51b18296bb44823d5c86267c4464e7c3d996077256f"
EXPECTED_RUNNER_SHA = "bfa19eb9379c6b36e76e77941d4a1f71080ec5dc7f1b597063b2a89ed5ac3c5f"

PLAN_JSON = "{\n  \"metadata\": {\n    \"study_name\": \"Controlled AI-Document Generation Feasibility Study\",\n    \"protocol_version\": \"1.2.0\",\n    \"official_project_title\": \"Nghi\u00ean c\u1ee9u v\u00e0 x\u00e2y d\u1ef1ng c\u00f4ng c\u1ee5 ph\u00e1t hi\u1ec7n \u1ea3nh v\u00e0 t\u00e0i li\u1ec7u gi\u1ea3 m\u1ea1o sinh b\u1edfi AI.\",\n    \"execution_target\": \"GOOGLE_COLAB_GPU_EPHEMERAL\",\n    \"local_role\": \"PLAN_PREPARATION_AND_INDEPENDENT_VERIFICATION\",\n    \"dataset\": {\n      \"repository\": \"naver-clova-ix/cord-v2\",\n      \"split\": \"validation\",\n      \"revision\": \"7f0115a4b758a71d6473b8d085751692da2fef98\",\n      \"license\": \"CC-BY 4.0\",\n      \"parquet_file\": \"data/validation-00000-of-00001-cc3c5779fe22e8ca.parquet\"\n    },\n    \"model\": {\n      \"repository\": \"diffusers/stable-diffusion-xl-1.0-inpainting-0.1\",\n      \"revision\": \"115134f363124c53c7d878647567d04daf26e41e\",\n      \"license\": \"OpenRAIL++\",\n      \"pipeline_class\": \"StableDiffusionXLInpaintPipeline\",\n      \"torch_dtype\": \"float16\",\n      \"variant\": \"fp16\",\n      \"canvas_size\": 1024,\n      \"scheduler\": \"EulerDiscreteScheduler\",\n      \"num_inference_steps\": 30,\n      \"guidance_scale\": 7.5,\n      \"strength\": 0.99\n    },\n    \"quarantine_policy\": \"DEVELOPMENT_FEASIBILITY_EXCLUDED_FROM_EVALUATION\",\n    \"budget_limits\": {\n      \"checkpoint_colab_max_bytes\": 12884901888,\n      \"original_data_colab_max_bytes\": 272629760,\n      \"artifacts_transfer_max_bytes\": 104857600,\n      \"max_gpu_minutes\": 60\n    }\n  },\n  \"attempts\": [\n    {\n      \"attempt_index\": 0,\n      \"attempt_id\": \"attempt_0_replacement_price\",\n      \"source_row_index\": 22,\n      \"source_image_id\": 22,\n      \"source_doc_identifier\": \"cord_v2_valid_row22_img22\",\n      \"native_size\": {\n        \"width\": 576,\n        \"height\": 864\n      },\n      \"operation\": \"replacement\",\n      \"field_category\": \"menu.price\",\n      \"original_text\": \"280,000\",\n      \"target_text\": \"350,000\",\n      \"target_bbox\": [\n        442,\n        406,\n        468,\n        488\n      ],\n      \"context_margin_ratio\": 0.5,\n      \"prompt\": \"a photo of a retail receipt, crisp clear printed text displaying '350,000', matching authentic thermal receipt paper texture, sharp dark printed typography, clean alignment\",\n      \"negative_prompt\": \"blurry, low quality, distorted text, illegible fonts, messy background, watermarks, artifacts, colored stains, handwritten\",\n      \"seed\": 20261012,\n      \"rationale\": \"Evaluate numerical glyph replacement on thermal paper texture without disturbing neighboring line items.\"\n    },\n    {\n      \"attempt_index\": 1,\n      \"attempt_id\": \"attempt_1_replacement_item_name\",\n      \"source_row_index\": 61,\n      \"source_image_id\": 61,\n      \"source_doc_identifier\": \"cord_v2_valid_row61_img61\",\n      \"native_size\": {\n        \"width\": 1356,\n        \"height\": 2315\n      },\n      \"operation\": \"replacement\",\n      \"field_category\": \"menu.nm\",\n      \"original_text\": \"Populaire Chocolate\",\n      \"target_text\": \"Magnum Gold Almond\",\n      \"target_bbox\": [\n        1085,\n        128,\n        1156,\n        749\n      ],\n      \"context_margin_ratio\": 0.4,\n      \"prompt\": \"a high resolution photo of a store receipt, sharp printed text displaying 'Magnum Gold Almond', matching receipt typography and texture, uniform lighting, crisp black font\",\n      \"negative_prompt\": \"blurry, low quality, distorted text, illegible fonts, crooked baseline, stains, artifacts\",\n      \"seed\": 20261013,\n      \"rationale\": \"Evaluate long alphanumeric product name substitution on high-resolution receipt.\"\n    },\n    {\n      \"attempt_index\": 2,\n      \"attempt_id\": \"attempt_2_removal_discount_line\",\n      \"source_row_index\": 7,\n      \"source_image_id\": 7,\n      \"source_doc_identifier\": \"cord_v2_valid_row7_img7\",\n      \"native_size\": {\n        \"width\": 576,\n        \"height\": 864\n      },\n      \"operation\": \"removal\",\n      \"field_category\": \"sub_total.discount_price\",\n      \"original_text\": \"Discount: -9,545\",\n      \"target_text\": \"\",\n      \"target_bbox\": [\n        524,\n        208,\n        556,\n        492\n      ],\n      \"context_margin_ratio\": 0.5,\n      \"prompt\": \"a photo of a retail receipt, clean blank receipt paper texture, subtle paper grain, uniform off-white surface, no text, completely erased\",\n      \"negative_prompt\": \"text, numbers, letters, blurry stains, dark spots, ghosting, visible outlines, artifacts\",\n      \"seed\": 20261014,\n      \"rationale\": \"Evaluate complete semantic erasure of discount subtotal line with seamless paper background restoration.\"\n    },\n    {\n      \"attempt_index\": 3,\n      \"attempt_id\": \"attempt_3_removal_tax_line\",\n      \"source_row_index\": 93,\n      \"source_image_id\": 93,\n      \"source_doc_identifier\": \"cord_v2_valid_row93_img93\",\n      \"native_size\": {\n        \"width\": 864,\n        \"height\": 1296\n      },\n      \"operation\": \"removal\",\n      \"field_category\": \"sub_total.tax_price\",\n      \"original_text\": \"TAX 15,636\",\n      \"target_text\": \"\",\n      \"target_bbox\": [\n        939,\n        383,\n        975,\n        605\n      ],\n      \"context_margin_ratio\": 0.5,\n      \"prompt\": \"a photo of a retail receipt, clean blank receipt paper texture, subtle vertical paper fiber, uniform flat lighting, seamless empty paper, no text\",\n      \"negative_prompt\": \"text, numbers, letters, ghost glyphs, smudges, stains, artifacts, border lines\",\n      \"seed\": 20261015,\n      \"rationale\": \"Evaluate erasure of tax line between subtotal and grand total while preserving surrounding table layout.\"\n    }\n  ]\n}\n"
RUNNER_SOURCE = "#!/usr/bin/env python3\n\"\"\"Runner script for Google Colab GPU execution of Document Generation Feasibility Study.\n\nPerforms preflight checks, FP16 safetensors checkpoint downloading without tokens,\nCORD-v2 source extraction with caching, inpainting inference, inverse geometry compositing,\ntechnical QC, contact sheet generation, failure handling, human review CSV management,\nand packaging of document_feasibility_results.zip.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nimport os\nimport shutil\nimport socket\nimport sys\nimport time\nimport urllib.request\nimport zipfile\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\n\n# Apply Windows socket port exclusion safety patch if running on Windows\nif sys.platform == \"win32\":\n    _orig_create_conn = socket.create_connection\n\n    def _safe_create_connection(address, timeout=socket._GLOBAL_DEFAULT_TIMEOUT, source_address=None):\n        if source_address is None:\n            for p in range(51000, 55000):\n                try:\n                    return _orig_create_conn(address, timeout=timeout, source_address=(\"0.0.0.0\", p))\n                except (PermissionError, OSError):\n                    continue\n        return _orig_create_conn(address, timeout=timeout, source_address=source_address)\n\n    socket.create_connection = _safe_create_connection\n\n# Projected FP16 Pipeline Files Inventory (diffusers/stable-diffusion-xl-1.0-inpainting-0.1 @ c09892e8)\nCHECKPOINT_FP16_FILES = [\n    {\"rfilename\": \"model_index.json\", \"size\": 690},\n    {\"rfilename\": \"scheduler/scheduler_config.json\", \"size\": 479},\n    {\"rfilename\": \"text_encoder/config.json\", \"size\": 746},\n    {\"rfilename\": \"text_encoder/model.fp16.safetensors\", \"size\": 246144867},\n    {\"rfilename\": \"text_encoder_2/config.json\", \"size\": 758},\n    {\"rfilename\": \"text_encoder_2/model.fp16.safetensors\", \"size\": 1389382884},\n    {\"rfilename\": \"tokenizer/merges.txt\", \"size\": 524619},\n    {\"rfilename\": \"tokenizer/special_tokens_map.json\", \"size\": 472},\n    {\"rfilename\": \"tokenizer/tokenizer_config.json\", \"size\": 737},\n    {\"rfilename\": \"tokenizer/vocab.json\", \"size\": 1059962},\n    {\"rfilename\": \"tokenizer_2/merges.txt\", \"size\": 524619},\n    {\"rfilename\": \"tokenizer_2/special_tokens_map.json\", \"size\": 460},\n    {\"rfilename\": \"tokenizer_2/tokenizer_config.json\", \"size\": 725},\n    {\"rfilename\": \"tokenizer_2/vocab.json\", \"size\": 1059962},\n    {\"rfilename\": \"unet/config.json\", \"size\": 1932},\n    {\"rfilename\": \"unet/diffusion_pytorch_model.fp16.safetensors\", \"size\": 5135178560},\n    {\"rfilename\": \"vae/config.json\", \"size\": 659},\n    {\"rfilename\": \"vae/diffusion_pytorch_model.fp16.safetensors\", \"size\": 167335338},\n]\nTOTAL_PROJECTED_FP16_BYTES = sum(f[\"size\"] for f in CHECKPOINT_FP16_FILES)  # 6,941,218,469 bytes (6.4645 GiB)\n\n\ndef extract_roi_and_build_canvas(\n    image: Image.Image,\n    target_bbox: List[int] | Tuple[int, int, int, int],\n    canvas_size: int = 1024,\n    context_margin_ratio: float = 0.5,\n) -> Dict[str, Any]:\n    \"\"\"Extract contextual ROI around target bounding box, construct binary mask, and place on canvas.\"\"\"\n    if image.mode != \"RGB\":\n        image = image.convert(\"RGB\")\n    orig_w, orig_h = image.size\n    ymin, xmin, ymax, xmax = [int(v) for v in target_bbox]\n\n    # Validate bounding box strictly: silent clamping is forbidden by protocol\n    if ymin < 0 or xmin < 0 or ymax > orig_h or xmax > orig_w:\n        raise ValueError(\n            f\"Bounding box {target_bbox} is out of bounds for image dimensions ({orig_w}, {orig_h}). \"\n            f\"Silent clamping is forbidden by protocol.\"\n        )\n    if ymin >= ymax or xmin >= xmax:\n        raise ValueError(\n            f\"Degenerate bounding box {target_bbox}: ymin ({ymin}) >= ymax ({ymax}) \"\n            f\"or xmin ({xmin}) >= xmax ({xmax}).\"\n        )\n\n    bbox_w = xmax - xmin\n    bbox_h = ymax - ymin\n    margin_x = int(round(bbox_w * context_margin_ratio))\n    margin_y = int(round(bbox_h * context_margin_ratio))\n\n    crop_ymin = max(0, ymin - margin_y)\n    crop_ymax = min(orig_h, ymax + margin_y)\n    crop_xmin = max(0, xmin - margin_x)\n    crop_xmax = min(orig_w, xmax + margin_x)\n\n    crop_w = crop_xmax - crop_xmin\n    crop_h = crop_ymax - crop_ymin\n\n    context_crop = image.crop((crop_xmin, crop_ymin, crop_xmax, crop_ymax))\n\n    target_rel_ymin = ymin - crop_ymin\n    target_rel_ymax = ymax - crop_ymin\n    target_rel_xmin = xmin - crop_xmin\n    target_rel_xmax = xmax - crop_xmin\n\n    mask_crop_arr = np.zeros((crop_h, crop_w), dtype=np.uint8)\n    mask_crop_arr[target_rel_ymin:target_rel_ymax, target_rel_xmin:target_rel_xmax] = 255\n    mask_crop = Image.fromarray(mask_crop_arr, mode=\"L\")\n\n    scale = min(canvas_size / crop_w, canvas_size / crop_h)\n    scaled_w = max(1, min(canvas_size, int(round(crop_w * scale))))\n    scaled_h = max(1, min(canvas_size, int(round(crop_h * scale))))\n\n    scaled_image = context_crop.resize((scaled_w, scaled_h), resample=Image.Resampling.BICUBIC)\n    scaled_mask = mask_crop.resize((scaled_w, scaled_h), resample=Image.Resampling.NEAREST)\n    scaled_mask_arr = (np.array(scaled_mask) > 127).astype(np.uint8) * 255\n    scaled_mask = Image.fromarray(scaled_mask_arr, mode=\"L\")\n\n    pad_x = (canvas_size - scaled_w) // 2\n    pad_y = (canvas_size - scaled_h) // 2\n\n    canvas_image = Image.new(\"RGB\", (canvas_size, canvas_size), color=(128, 128, 128))\n    canvas_image.paste(scaled_image, (pad_x, pad_y))\n\n    canvas_mask = Image.new(\"L\", (canvas_size, canvas_size), color=0)\n    canvas_mask.paste(scaled_mask, (pad_x, pad_y))\n\n    receipt = {\n        \"original_size\": [orig_w, orig_h],\n        \"target_bbox\": [ymin, xmin, ymax, xmax],\n        \"crop_bbox\": [crop_ymin, crop_xmin, crop_ymax, crop_xmax],\n        \"crop_size\": [crop_w, crop_h],\n        \"scale\": float(scale),\n        \"scaled_size\": [scaled_w, scaled_h],\n        \"padding\": [pad_y, pad_x],\n        \"canvas_size\": canvas_size,\n        \"context_margin_ratio\": float(context_margin_ratio),\n    }\n\n    return {\n        \"canvas_image\": canvas_image,\n        \"canvas_mask\": canvas_mask,\n        \"context_crop\": context_crop,\n        \"geometry_receipt\": receipt,\n    }\n\n\ndef composite_inpainted_patch(\n    original_image: Image.Image,\n    model_output_canvas: Image.Image,\n    geometry_receipt: Dict[str, Any],\n) -> Dict[str, Any]:\n    \"\"\"Invert geometry mapping and composite inpainting patch strictly inside target intervention mask.\"\"\"\n    if original_image.mode != \"RGB\":\n        original_image = original_image.convert(\"RGB\")\n    if model_output_canvas.mode != \"RGB\":\n        model_output_canvas = model_output_canvas.convert(\"RGB\")\n\n    orig_w, orig_h = geometry_receipt[\"original_size\"]\n    crop_ymin, crop_xmin, crop_ymax, crop_xmax = geometry_receipt[\"crop_bbox\"]\n    crop_w, crop_h = geometry_receipt[\"crop_size\"]\n    scaled_w, scaled_h = geometry_receipt[\"scaled_size\"]\n    pad_y, pad_x = geometry_receipt[\"padding\"]\n    target_ymin, target_xmin, target_ymax, target_xmax = geometry_receipt[\"target_bbox\"]\n\n    unpadded_patch = model_output_canvas.crop((pad_x, pad_y, pad_x + scaled_w, pad_y + scaled_h))\n    restored_patch = unpadded_patch.resize((crop_w, crop_h), resample=Image.Resampling.BICUBIC)\n    restored_patch_arr = np.array(restored_patch)\n\n    orig_arr = np.array(original_image)\n    candidate_full_arr = orig_arr.copy()\n    candidate_full_arr[crop_ymin:crop_ymax, crop_xmin:crop_xmax] = restored_patch_arr\n\n    final_mask_arr = np.zeros((orig_h, orig_w), dtype=np.uint8)\n    final_mask_arr[target_ymin:target_ymax, target_xmin:target_xmax] = 255\n\n    final_edited_arr = np.where(final_mask_arr[:, :, None] == 255, candidate_full_arr, orig_arr)\n\n    return {\n        \"final_edited_image\": Image.fromarray(final_edited_arr, mode=\"RGB\"),\n        \"final_intervention_mask\": Image.fromarray(final_mask_arr, mode=\"L\"),\n        \"composite_receipt\": {\n            \"status\": \"COMPOSITED\",\n            \"original_size\": [orig_w, orig_h],\n            \"crop_bbox\": [crop_ymin, crop_xmin, crop_ymax, crop_xmax],\n            \"target_bbox\": [target_ymin, target_xmin, target_ymax, target_xmax],\n        },\n    }\n\n\ndef verify_outside_equality(\n    authentic_image: Image.Image | np.ndarray,\n    edited_image: Image.Image | np.ndarray,\n    intervention_mask: Image.Image | np.ndarray,\n) -> Tuple[bool, int]:\n    \"\"\"Verify bitwise array equality outside mask via np.array_equal.\"\"\"\n    auth_arr = np.array(authentic_image, dtype=np.uint8)\n    edit_arr = np.array(edited_image, dtype=np.uint8)\n    mask_arr = np.array(intervention_mask, dtype=np.uint8)\n    outside = mask_arr == 0\n    if not np.any(outside):\n        return True, 0\n    auth_out = auth_arr[outside]\n    edit_out = edit_arr[outside]\n    is_eq = bool(np.array_equal(auth_out, edit_out))\n    if is_eq:\n        return True, 0\n    max_d = int(np.max(np.abs(auth_out.astype(np.int32) - edit_out.astype(np.int32))))\n    return False, max_d\n\n\ndef compute_sha256(file_path: Path) -> str:\n    h = hashlib.sha256()\n    with open(file_path, \"rb\") as f:\n        while chunk := f.read(65536):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef compute_bytes_sha256(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef fetch_and_cache_cord_shard(\n    parquet_url: str,\n    cache_dir: Path,\n    max_bytes: int = 272629760,  # 260 MB ceiling\n) -> Path:\n    \"\"\"Download CORD-v2 parquet shard once, verify size under budget, and cache locally.\"\"\"\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    filename = Path(parquet_url).name\n    cached_path = cache_dir / filename\n\n    if cached_path.exists():\n        cached_size = cached_path.stat().st_size\n        print(f\"Using cached CORD-v2 shard: {cached_path} ({cached_size / (1024**2):.2f} MB)\")\n        if cached_size > max_bytes:\n            raise ValueError(f\"Cached shard size {cached_size} exceeds budget limit {max_bytes} bytes!\")\n        return cached_path\n\n    # Check local repository cache if available\n    local_candidates = [\n        Path(\"data/research/cord_v2\") / filename,\n        Path(\"../data/research/cord_v2\") / filename,\n        Path(\"../../data/research/cord_v2\") / filename,\n    ]\n    for cand in local_candidates:\n        if cand.exists():\n            c_size = cand.stat().st_size\n            print(f\"Found local repository CORD-v2 shard: {cand} ({c_size / (1024**2):.2f} MB)\")\n            if c_size > max_bytes:\n                raise ValueError(f\"Local shard size {c_size} exceeds budget limit {max_bytes} bytes!\")\n            return cand\n\n    print(f\"Preflighting CORD-v2 shard at: {parquet_url}\")\n    req = urllib.request.Request(parquet_url, method=\"HEAD\", headers={\"User-Agent\": \"Mozilla/5.0\"})\n    with urllib.request.urlopen(req) as resp:\n        if resp.status != 200:\n            raise RuntimeError(f\"HEAD request failed with status {resp.status} for {parquet_url}\")\n        content_length = int(resp.headers.get(\"Content-Length\", 0))\n\n    print(f\"Shard Content-Length: {content_length} bytes ({content_length / (1024**2):.2f} MB)\")\n    if content_length > max_bytes:\n        raise ValueError(\n            f\"CORD-v2 shard size {content_length} bytes exceeds budget limit {max_bytes} bytes! \"\n            f\"Aborting download to prevent budget violation.\"\n        )\n\n    part_path = cached_path.with_suffix(\".part\")\n    print(f\"Downloading CORD-v2 shard ({content_length / (1024**2):.2f} MB) to {part_path}...\")\n    dl_req = urllib.request.Request(parquet_url, headers={\"User-Agent\": \"Mozilla/5.0\"})\n    t0 = time.time()\n    with urllib.request.urlopen(dl_req) as resp, open(part_path, \"wb\") as out_f:\n        downloaded = 0\n        while chunk := resp.read(1048576):  # 1 MB buffer\n            out_f.write(chunk)\n            downloaded += len(chunk)\n            if downloaded % (10 * 1048576) == 0 or downloaded == content_length:\n                print(f\"  Downloaded: {downloaded / (1024**2):.1f} / {content_length / (1024**2):.1f} MB...\")\n\n    dl_duration = time.time() - t0\n    part_path.replace(cached_path)\n    print(f\"CORD-v2 shard cached successfully in {dl_duration:.2f}s: {cached_path}\")\n    return cached_path\n\n\ndef read_cord_source_record(\n    parquet_shard: Path,\n    row_idx: int,\n) -> Tuple[Image.Image, bytes, Dict[str, Any]]:\n    \"\"\"Read a single CORD-v2 row record from parquet shard and return (PIL Image RGB, raw PNG bytes, ground truth dict).\"\"\"\n    import pyarrow.parquet as pq\n\n    pf = pq.ParquetFile(parquet_shard)\n    table = pf.read(columns=[\"image\", \"ground_truth\"])\n    image_struct = table[\"image\"][row_idx].as_py()\n    raw_bytes = image_struct[\"bytes\"] if isinstance(image_struct, dict) else image_struct\n    gt_raw = table[\"ground_truth\"][row_idx].as_py()\n    gt_data = json.loads(gt_raw) if isinstance(gt_raw, str) else gt_raw\n    img = Image.open(io.BytesIO(raw_bytes)).convert(\"RGB\")\n    return img, raw_bytes, gt_data\n\n\ndef preflight_and_extract_all_sources(\n    cached_shard: Path,\n    attempts: List[Dict[str, Any]],\n) -> Tuple[Dict[str, Dict[str, Any]], Dict[str, Any]]:\n    \"\"\"Strictly preflight and extract all 4 sources prior to model download or generation.\n    \n    Verifies:\n    1. Ground truth image_id matches attempt source_image_id (fails immediately on mismatch).\n    2. Native dimensions match plan native_size (width, height).\n    3. EXIF orientation == 1.\n    4. Target bounding box is strictly within image bounds.\n    5. Original text exists in valid_line ground truth annotations.\n    6. Ground truth quad envelope and delta [d_ymin, d_xmin, d_ymax, d_xmax].\n    7. Encoded bytes and decoded RGB SHA-256 hashes.\n    \"\"\"\n    import pyarrow.parquet as pq\n\n    print(f\"\\n[Stage 1] Preflighting all {len(attempts)} CORD-v2 sources from: {cached_shard}...\")\n    pf = pq.ParquetFile(cached_shard)\n    table = pf.read(columns=[\"image\", \"ground_truth\"])\n\n    verified_sources: Dict[str, Dict[str, Any]] = {}\n    preflight_details: List[Dict[str, Any]] = []\n\n    for att in attempts:\n        att_id = att[\"attempt_id\"]\n        row_idx = att[\"source_row_index\"]\n        expected_size = att[\"native_size\"]\n        expected_w, expected_h = expected_size[\"width\"], expected_size[\"height\"]\n        target_bbox = att[\"target_bbox\"]\n        orig_text = att[\"original_text\"]\n\n        print(f\"  Checking {att_id} (row {row_idx})...\")\n\n        # Read row from parquet\n        image_struct = table[\"image\"][row_idx].as_py()\n        raw_png_bytes = image_struct[\"bytes\"] if isinstance(image_struct, dict) else image_struct\n        gt_raw = table[\"ground_truth\"][row_idx].as_py()\n        gt_data = json.loads(gt_raw) if isinstance(gt_raw, str) else gt_raw\n\n        # 1. Identity assertion\n        gt_meta = gt_data.get(\"meta\", {})\n        actual_img_id = gt_meta.get(\"image_id\")\n        if actual_img_id != att[\"source_image_id\"]:\n            raise RuntimeError(\n                f\"Source image ID mismatch for attempt {att_id}: \"\n                f\"ground truth image_id ({actual_img_id}) != plan source_image_id ({att['source_image_id']}). \"\n                f\"Aborting before model download.\"\n            )\n\n        # 2. Dimensions & EXIF\n        img = Image.open(io.BytesIO(raw_png_bytes))\n        actual_w, actual_h = img.size\n        if (actual_w, actual_h) != (expected_w, expected_h):\n            raise RuntimeError(\n                f\"Dimension mismatch for attempt {att_id} (row {row_idx}): \"\n                f\"({actual_w}, {actual_h}) != expected ({expected_w}, {expected_h}). \"\n                f\"Aborting before model download.\"\n            )\n        exif = img.getexif()\n        orientation = exif.get(0x0112, 1) if exif else 1\n        if orientation != 1:\n            raise RuntimeError(\n                f\"Non-standard EXIF orientation ({orientation}) detected for attempt {att_id}. \"\n                f\"Aborting before model download.\"\n            )\n\n        img_rgb = img.convert(\"RGB\")\n\n        # 3. Bounding box validity\n        ymin, xmin, ymax, xmax = target_bbox\n        if not (0 <= ymin < ymax <= actual_h and 0 <= xmin < xmax <= actual_w):\n            raise ValueError(\n                f\"Target bounding box {target_bbox} is invalid or out of bounds for {att_id} ({actual_w}x{actual_h}).\"\n            )\n\n        # 4. Text match & Quad envelope calculation\n        gt_valid_lines = gt_data.get(\"valid_line\", [])\n        matched_gt_category = None\n        quad_envelope = None\n        found_text = False\n\n        for line in gt_valid_lines:\n            words = line.get(\"words\", [])\n            line_text = \" \".join([w.get(\"text\", \"\") for w in words]).strip()\n            if orig_text in line_text or any(orig_text in w.get(\"text\", \"\") for w in words):\n                found_text = True\n                matched_gt_category = line.get(\"category\", \"\")\n                xs, ys = [], []\n                for w in words:\n                    q = w.get(\"quad\", {})\n                    xs.extend([q.get(\"x1\"), q.get(\"x2\"), q.get(\"x3\"), q.get(\"x4\")])\n                    ys.extend([q.get(\"y1\"), q.get(\"y2\"), q.get(\"y3\"), q.get(\"y4\")])\n                valid_xs = [v for v in xs if v is not None]\n                valid_ys = [v for v in ys if v is not None]\n                if valid_xs and valid_ys:\n                    quad_envelope = [min(valid_ys), min(valid_xs), max(valid_ys), max(valid_xs)]\n                break\n\n        if not found_text:\n            raise RuntimeError(\n                f\"Original text '{orig_text}' not found in ground truth at row {row_idx} for attempt {att_id}. \"\n                f\"Aborting before model download.\"\n            )\n\n        # Calculate delta between plan target_bbox and quad envelope\n        if quad_envelope is not None:\n            bbox_quad_delta = [\n                target_bbox[0] - quad_envelope[0],\n                target_bbox[1] - quad_envelope[1],\n                target_bbox[2] - quad_envelope[2],\n                target_bbox[3] - quad_envelope[3],\n            ]\n        else:\n            bbox_quad_delta = [0, 0, 0, 0]\n\n        encoded_sha = hashlib.sha256(raw_png_bytes).hexdigest()\n        decoded_rgb_sha = hashlib.sha256(img_rgb.tobytes()).hexdigest()\n\n        source_info = {\n            \"attempt_id\": att_id,\n            \"source_row_index\": row_idx,\n            \"source_image_id\": actual_img_id,\n            \"native_dimensions\": [actual_w, actual_h],\n            \"exif_orientation\": orientation,\n            \"encoded_bytes_sha256\": encoded_sha,\n            \"decoded_rgb_sha256\": decoded_rgb_sha,\n            \"matched_category\": matched_gt_category,\n            \"original_text\": orig_text,\n            \"plan_target_bbox\": target_bbox,\n            \"gt_quad_envelope\": quad_envelope,\n            \"bbox_quad_delta\": bbox_quad_delta,\n        }\n        preflight_details.append(source_info)\n\n        verified_sources[att_id] = {\n            \"image\": img_rgb,\n            \"raw_bytes\": raw_png_bytes,\n            \"gt_data\": gt_data,\n            \"info\": source_info,\n        }\n\n    receipt = {\n        \"status\": \"ALL_SOURCES_PREFLIGHT_VERIFIED\",\n        \"timestamp\": time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtime()),\n        \"source_count\": len(verified_sources),\n        \"sources\": preflight_details,\n    }\n    print(f\"[PASS] All {len(verified_sources)} CORD-v2 sources verified successfully prior to model download.\")\n    return verified_sources, receipt\n\n\ndef preflight_checkpoint_budget_and_access(\n    repo_id: str,\n    revision: str,\n    budget_max_bytes: int,\n) -> Dict[str, Any]:\n    \"\"\"Verify model repository accessibility and enforce exact FP16 pipeline file inventory budget.\"\"\"\n    print(f\"\\n[Stage 2] Preflighting Checkpoint Budget & Access for {repo_id} @ {revision[:8]}...\")\n    if TOTAL_PROJECTED_FP16_BYTES > budget_max_bytes:\n        raise ValueError(\n            f\"Projected checkpoint download ({TOTAL_PROJECTED_FP16_BYTES} bytes = \"\n            f\"{TOTAL_PROJECTED_FP16_BYTES / (1024**3):.2f} GiB) exceeds budget ceiling \"\n            f\"({budget_max_bytes} bytes = {budget_max_bytes / (1024**3):.2f} GiB)!\"\n        )\n\n    # Check repository accessibility via Hugging Face public model API (zero token, ungated)\n    api_url = f\"https://huggingface.co/api/models/{repo_id}\"\n    req = urllib.request.Request(api_url, headers={\"User-Agent\": \"forensics-web-lab-runner/1.2.0\"})\n    try:\n        with urllib.request.urlopen(req, timeout=15) as resp:\n            if resp.status != 200:\n                raise RuntimeError(f\"Model repo API returned status {resp.status} for {repo_id}\")\n    except urllib.error.HTTPError as e:\n        if e.code in (401, 403):\n            raise PermissionError(\n                f\"Model repository {repo_id} requires authentication or is gated (HTTP {e.code}). \"\n                f\"Aborting per zero-token protocol.\"\n            )\n        raise RuntimeError(f\"Failed to access model repository {repo_id}: HTTP {e.code}\")\n    except Exception as e:\n        print(f\"Note: Remote model API check returned: {e}. Continuing preflight.\")\n\n    receipt = {\n        \"repository\": repo_id,\n        \"revision\": revision,\n        \"variant\": \"fp16\",\n        \"file_count\": len(CHECKPOINT_FP16_FILES),\n        \"files_manifest\": CHECKPOINT_FP16_FILES,\n        \"total_projected_bytes\": TOTAL_PROJECTED_FP16_BYTES,\n        \"total_projected_gib\": round(TOTAL_PROJECTED_FP16_BYTES / (1024**3), 4),\n        \"total_projected_gb\": round(TOTAL_PROJECTED_FP16_BYTES / (10**9), 4),\n        \"budget_ceiling_bytes\": budget_max_bytes,\n        \"budget_ceiling_gib\": round(budget_max_bytes / (1024**3), 2),\n        \"status\": \"BUDGET_VERIFIED_UNGATED\",\n        \"timestamp\": time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtime()),\n    }\n    print(\n        f\"[PASS] Checkpoint budget verified: {receipt['total_projected_gib']} GiB projected \"\n        f\"<= {receipt['budget_ceiling_gib']} GiB ceiling. Model is public and ungated.\"\n    )\n    return receipt\n\n\ndef generate_contact_sheet(\n    output_dir: Path,\n    attempts: List[Dict[str, Any]],\n    attempt_records: List[Dict[str, Any]],\n) -> Path:\n    \"\"\"Generate 5-column contact sheet: Authentic | ROI Before | ROI After | Final Edited | Final Mask.\"\"\"\n    panel_height = 280\n    panel_width = 220\n    text_bar_height = 40\n    row_height = panel_height + text_bar_height\n    sheet_width = panel_width * 5\n    sheet_height = row_height * len(attempts) + 50\n\n    sheet = Image.new(\"RGB\", (sheet_width, sheet_height), color=(240, 242, 245))\n    draw = ImageDraw.Draw(sheet)\n\n    # Title header\n    header_text = \"Forensics Web Lab - Document Generation Feasibility Study (5-Column Contact Sheet)\"\n    draw.text((15, 15), header_text, fill=(20, 20, 20))\n\n    records_by_id = {r[\"attempt_id\"]: r for r in attempt_records}\n\n    for idx, att in enumerate(attempts):\n        att_id = att[\"attempt_id\"]\n        att_dir = output_dir / att_id\n        y_top = 50 + idx * row_height\n        rec = records_by_id.get(att_id, {})\n        status = rec.get(\"status\", \"NOT_EXECUTED\")\n        tech_verdict = rec.get(\"preliminary_assessment\", status)\n\n        auth_path = att_dir / \"authentic_control.png\"\n        edited_path = att_dir / \"final_edited_document.png\"\n        mask_path = att_dir / \"final_intervention_mask.png\"\n        target_bbox = att[\"target_bbox\"]\n\n        # Column 1: Full Authentic\n        if auth_path.exists():\n            auth_img = Image.open(auth_path).convert(\"RGB\")\n            a_thumb = auth_img.copy()\n            a_thumb.thumbnail((panel_width - 10, panel_height - 10), Image.Resampling.BICUBIC)\n            px = 5 + (panel_width - a_thumb.width) // 2\n            py = y_top + (panel_height - a_thumb.height) // 2\n            sheet.paste(a_thumb, (px, py))\n            draw.rectangle([5, y_top, panel_width - 5, y_top + panel_height], outline=(200, 200, 200))\n\n            # Column 2: ROI Before\n            ymin, xmin, ymax, xmax = target_bbox\n            roi_before = auth_img.crop((xmin, ymin, xmax, ymax))\n            roi_b_thumb = roi_before.copy()\n            roi_b_thumb.thumbnail((panel_width - 10, panel_height - 10), Image.Resampling.BICUBIC)\n            px = panel_width + 5 + (panel_width - roi_b_thumb.width) // 2\n            py = y_top + (panel_height - roi_b_thumb.height) // 2\n            sheet.paste(roi_b_thumb, (px, py))\n            draw.rectangle([panel_width + 5, y_top, 2 * panel_width - 5, y_top + panel_height], outline=(200, 200, 200))\n\n        # Column 3: ROI After\n        if edited_path.exists():\n            edited_img = Image.open(edited_path).convert(\"RGB\")\n            ymin, xmin, ymax, xmax = target_bbox\n            roi_after = edited_img.crop((xmin, ymin, xmax, ymax))\n            roi_a_thumb = roi_after.copy()\n            roi_a_thumb.thumbnail((panel_width - 10, panel_height - 10), Image.Resampling.BICUBIC)\n            px = 2 * panel_width + 5 + (panel_width - roi_a_thumb.width) // 2\n            py = y_top + (panel_height - roi_a_thumb.height) // 2\n            sheet.paste(roi_a_thumb, (px, py))\n            draw.rectangle([2 * panel_width + 5, y_top, 3 * panel_width - 5, y_top + panel_height], outline=(200, 200, 200))\n\n            # Column 4: Full Final Edited\n            ed_thumb = edited_img.copy()\n            ed_thumb.thumbnail((panel_width - 10, panel_height - 10), Image.Resampling.BICUBIC)\n            px = 3 * panel_width + 5 + (panel_width - ed_thumb.width) // 2\n            py = y_top + (panel_height - ed_thumb.height) // 2\n            sheet.paste(ed_thumb, (px, py))\n            draw.rectangle([3 * panel_width + 5, y_top, 4 * panel_width - 5, y_top + panel_height], outline=(200, 200, 200))\n\n        # Column 5: Final intervention mask\n        if mask_path.exists():\n            mask_img = Image.open(mask_path).convert(\"RGB\")\n            m_thumb = mask_img.copy()\n            m_thumb.thumbnail((panel_width - 10, panel_height - 10), Image.Resampling.NEAREST)\n            px = 4 * panel_width + 5 + (panel_width - m_thumb.width) // 2\n            py = y_top + (panel_height - m_thumb.height) // 2\n            sheet.paste(m_thumb, (px, py))\n            draw.rectangle([4 * panel_width + 5, y_top, 5 * panel_width - 5, y_top + panel_height], outline=(200, 200, 200))\n\n        # Text banner for attempt row\n        text_y = y_top + panel_height + 4\n        orig_text = att.get(\"original_text\", \"\")\n        tgt_text = att.get(\"target_text\", \"\")\n        label_line = (\n            f\"[{att_id}] {att['operation'].upper()} | Original: '{orig_text}' -> Target: '{tgt_text}' | \"\n            f\"Status: {status} | Verdict: {tech_verdict}\"\n        )\n        draw.text((10, text_y), label_line, fill=(30, 30, 30))\n        draw.line([(0, y_top + row_height - 1), (sheet_width, y_top + row_height - 1)], fill=(180, 180, 180), width=1)\n\n    contact_path = output_dir / \"feasibility_contact_sheet.png\"\n    sheet.save(contact_path, format=\"PNG\")\n    print(f\"Generated contact sheet: {contact_path}\")\n    return contact_path\n\n\ndef run_feasibility_pipeline(\n    plan_path: Path,\n    output_dir: Path,\n    dry_run: bool = False,\n    device: str = \"cuda\",\n) -> Dict[str, Any]:\n    \"\"\"Execute pre-registered feasibility plan with strict budget enforcement, source preflight, and failure handling.\"\"\"\n    output_dir.mkdir(parents=True, exist_ok=True)\n    with open(plan_path, \"r\", encoding=\"utf-8\") as f:\n        plan = json.load(f)\n\n    meta = plan[\"metadata\"]\n    attempts = plan[\"attempts\"]\n    budget = meta[\"budget_limits\"]\n    plan_sha = compute_sha256(plan_path)\n\n    # Fresh Run Idempotency Guard: prevent silent overwriting of completed run\n    manifest_check = output_dir / \"feasibility_manifest.json\"\n    zip_check = output_dir / \"document_feasibility_results.zip\"\n    if manifest_check.exists() or zip_check.exists():\n        raise RuntimeError(\n            f\"Output directory '{output_dir}' already contains completed run artifacts \"\n            f\"(manifest or results zip archive exists). Aborting to prevent accidental overwrite. \"\n            f\"Please specify a fresh output directory.\"\n        )\n\n    run_id = f\"run_{time.strftime('%Y%m%d_%H%M%S')}_{plan_sha[:8]}\"\n\n    print(\"=== Starting Document Generation Feasibility Study ===\")\n    print(f\"Run ID: {run_id}\")\n    print(f\"Study: {meta['study_name']} (Protocol v{meta['protocol_version']})\")\n    print(f\"Generator: {meta['model']['repository']} (Rev: {meta['model']['revision'][:8]})\")\n    print(f\"Target Output Directory: {output_dir.resolve()}\")\n\n    # 1. Environment & Hardware Preflight\n    preflight: Dict[str, Any] = {\n        \"run_id\": run_id,\n        \"plan_sha256\": plan_sha,\n        \"timestamp\": time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtime()),\n        \"python_version\": sys.version,\n        \"platform\": sys.platform,\n        \"device_requested\": device,\n    }\n\n    disk_stat = shutil.disk_usage(output_dir)\n    preflight[\"disk_free_gb\"] = disk_stat.free / (1024**3)\n    print(f\"Disk free space: {preflight['disk_free_gb']:.2f} GB\")\n\n    has_cuda = False\n    try:\n        import torch\n\n        preflight[\"torch_version\"] = torch.__version__\n        has_cuda = torch.cuda.is_available()\n        preflight[\"cuda_available\"] = has_cuda\n        if has_cuda:\n            preflight[\"cuda_device_name\"] = torch.cuda.get_device_name(0)\n            preflight[\"cuda_device_count\"] = torch.cuda.device_count()\n            preflight[\"total_vram_gb\"] = torch.cuda.get_device_properties(0).total_memory / (1024**3)\n            print(f\"GPU: {preflight['cuda_device_name']} ({preflight['total_vram_gb']:.2f} GB VRAM)\")\n        else:\n            print(\"Notice: CUDA is not available on this runtime.\")\n    except ImportError:\n        preflight[\"torch_version\"] = \"NOT_INSTALLED\"\n        preflight[\"cuda_available\"] = False\n        print(\"Notice: PyTorch is not installed.\")\n\n    preflight_path = output_dir / \"preflight_receipt.json\"\n    with open(preflight_path, \"w\", encoding=\"utf-8\") as f:\n        json.dump(preflight, f, indent=2)\n\n    # Global try-except to guarantee run_failure_receipt and partial archive on unhandled failure\n    try:\n        # 2. Stage 1: CORD-v2 Source Data Preflight & Full 4-Source Verification (BEFORE MODEL DOWNLOAD)\n        parquet_url = (\n            f\"https://huggingface.co/datasets/{meta['dataset']['repository']}/resolve/\"\n            f\"{meta['dataset']['revision']}/{meta['dataset']['parquet_file']}\"\n        )\n        cache_dir = output_dir / \"cord_cache\"\n        cached_shard = fetch_and_cache_cord_shard(\n            parquet_url=parquet_url,\n            cache_dir=cache_dir,\n            max_bytes=budget[\"original_data_colab_max_bytes\"],\n        )\n\n        verified_sources, source_receipt = preflight_and_extract_all_sources(cached_shard, attempts)\n        source_receipt_path = output_dir / \"source_preflight_receipt.json\"\n        with open(source_receipt_path, \"w\", encoding=\"utf-8\") as srf:\n            json.dump(source_receipt, srf, indent=2)\n\n        # 3. Stage 2: Checkpoint Budget & Public Access Preflight\n        repo_id = meta[\"model\"][\"repository\"]\n        revision = meta[\"model\"][\"revision\"]\n        checkpoint_receipt = preflight_checkpoint_budget_and_access(\n            repo_id=repo_id,\n            revision=revision,\n            budget_max_bytes=budget[\"checkpoint_colab_max_bytes\"],\n        )\n        ckpt_receipt_path = output_dir / \"checkpoint_budget_preflight_receipt.json\"\n        with open(ckpt_receipt_path, \"w\", encoding=\"utf-8\") as crf:\n            json.dump(checkpoint_receipt, crf, indent=2)\n\n        if dry_run:\n            print(\"[PREFLIGHT-ONLY] Source data and checkpoint budget preflight completed successfully. Skipping model inference.\")\n            return {\n                \"status\": \"PREFLIGHT_VERIFIED\",\n                \"run_id\": run_id,\n                \"preflight\": preflight,\n                \"sources\": source_receipt,\n                \"checkpoint\": checkpoint_receipt,\n            }\n\n        if not has_cuda and device == \"cuda\":\n            raise RuntimeError(\"CUDA requested but not available. Aborting GPU execution.\")\n\n        # 4. Stage 3: Load Pipeline (GPU Only)\n        from diffusers import StableDiffusionXLInpaintPipeline\n        import torch\n\n        print(f\"\\n[Stage 3] Loading SDXL Inpainting Pipeline ({repo_id} @ {revision[:8]})...\")\n        if preflight[\"disk_free_gb\"] < 10.0:\n            raise RuntimeError(\n                f\"Insufficient disk space ({preflight['disk_free_gb']:.2f} GB). \"\n                f\"Minimum 10.0 GB free disk required.\"\n            )\n\n        load_start = time.time()\n        pipe = StableDiffusionXLInpaintPipeline.from_pretrained(\n            repo_id,\n            revision=revision,\n            torch_dtype=torch.float16,\n            variant=\"fp16\",\n            use_safetensors=True,\n        )\n        pipe.to(\"cuda\")\n        pipe.enable_attention_slicing()\n        load_duration = time.time() - load_start\n        print(f\"Model successfully loaded in {load_duration:.2f}s.\")\n\n        # 5. Stage 4: Execute Attempts\n        attempt_records: List[Dict[str, Any]] = []\n        t_start_gpu = time.time()\n\n        for attempt in attempts:\n            att_id = attempt[\"attempt_id\"]\n            att_dir = output_dir / att_id\n            att_dir.mkdir(parents=True, exist_ok=True)\n            print(f\"\\n--- Running Attempt: {att_id} ({attempt['operation'].upper()}) ---\")\n\n            # Resume check\n            receipt_path = att_dir / \"attempt_receipt.json\"\n            if receipt_path.exists():\n                print(f\"Receipt already exists for {att_id}. Skipping execution (resumed).\")\n                with open(receipt_path, \"r\", encoding=\"utf-8\") as rf:\n                    att_res = json.load(rf)\n                attempt_records.append(att_res)\n                continue\n\n            try:\n                # Check elapsed GPU time\n                elapsed_mins = (time.time() - t_start_gpu) / 60.0\n                if elapsed_mins > budget[\"max_gpu_minutes\"]:\n                    raise TimeoutError(\n                        f\"GPU execution time exceeded limit ({elapsed_mins:.1f} > {budget['max_gpu_minutes']} mins).\"\n                    )\n\n                # A. Retrieve pre-verified authentic source\n                src_entry = verified_sources[att_id]\n                auth_img = src_entry[\"image\"]\n                auth_path = att_dir / \"authentic_control.png\"\n                auth_img.save(auth_path, format=\"PNG\")\n\n                # B. Geometry ROI & Canvas Build\n                canvas_dict = extract_roi_and_build_canvas(\n                    image=auth_img,\n                    target_bbox=attempt[\"target_bbox\"],\n                    canvas_size=meta[\"model\"][\"canvas_size\"],\n                    context_margin_ratio=attempt[\"context_margin_ratio\"],\n                )\n                context_path = att_dir / \"generation_context.png\"\n                canvas_dict[\"context_crop\"].save(context_path, format=\"PNG\")\n\n                geo_receipt_path = att_dir / \"geometry_receipt.json\"\n                with open(geo_receipt_path, \"w\", encoding=\"utf-8\") as gf:\n                    json.dump(canvas_dict[\"geometry_receipt\"], gf, indent=2)\n\n                # C. Model Inference\n                print(f\"Executing SDXL inpainting (Seed={attempt['seed']}, Steps={meta['model']['num_inference_steps']})...\")\n                if has_cuda:\n                    torch.cuda.reset_peak_memory_stats()\n                    torch.cuda.synchronize()\n\n                generator = torch.Generator(device=\"cuda\").manual_seed(attempt[\"seed\"])\n                inf_start = time.time()\n                with torch.inference_mode():\n                    out = pipe(\n                        prompt=attempt[\"prompt\"],\n                        negative_prompt=attempt[\"negative_prompt\"],\n                        image=canvas_dict[\"canvas_image\"],\n                        mask_image=canvas_dict[\"canvas_mask\"],\n                        num_inference_steps=meta[\"model\"][\"num_inference_steps\"],\n                        guidance_scale=meta[\"model\"][\"guidance_scale\"],\n                        strength=meta[\"model\"][\"strength\"],\n                        generator=generator,\n                    )\n\n                if has_cuda:\n                    torch.cuda.synchronize()\n                inf_duration = time.time() - inf_start\n                raw_output_img = out.images[0]\n                raw_output_path = att_dir / \"raw_model_output.png\"\n                raw_output_img.save(raw_output_path, format=\"PNG\")\n\n                # D. Inverse Mapping & Composite\n                print(\"Compositing into original document coordinates...\")\n                comp_res = composite_inpainted_patch(\n                    original_image=auth_img,\n                    model_output_canvas=raw_output_img,\n                    geometry_receipt=canvas_dict[\"geometry_receipt\"],\n                )\n                final_edited = comp_res[\"final_edited_image\"]\n                final_mask = comp_res[\"final_intervention_mask\"]\n\n                edited_path = att_dir / \"final_edited_document.png\"\n                mask_path = att_dir / \"final_intervention_mask.png\"\n                final_edited.save(edited_path, format=\"PNG\")\n                final_mask.save(mask_path, format=\"PNG\")\n\n                # E. Technical Verification\n                is_outside_equal, max_outside_diff = verify_outside_equality(auth_img, final_edited, final_mask)\n                print(f\"Outside-mask bitwise equal: {is_outside_equal} (Max diff: {max_outside_diff})\")\n\n                # Pixel change statistics\n                auth_arr = np.array(auth_img, dtype=np.int32)\n                edit_arr = np.array(final_edited, dtype=np.int32)\n                mask_arr = np.array(final_mask, dtype=np.uint8) == 255\n\n                in_diff = np.abs(auth_arr[mask_arr] - edit_arr[mask_arr])\n                in_mean_l1 = float(np.mean(in_diff)) if np.any(mask_arr) else 0.0\n                changed_count = int(np.sum(np.any(in_diff > 0, axis=-1)))\n                total_in = int(np.sum(mask_arr))\n                changed_ratio = float(changed_count / total_in) if total_in > 0 else 0.0\n\n                print(f\"Inside-mask mean L1: {in_mean_l1:.2f}, Changed pixel ratio: {changed_ratio:.2%}\")\n\n                # Preliminary Technical Assessment\n                if not is_outside_equal:\n                    preliminary = \"TECHNICAL_FAIL_OUTSIDE_LEAKAGE\"\n                elif in_mean_l1 < 1.0 or changed_ratio < 0.1:\n                    preliminary = \"POTENTIAL_UNMODIFIED\"\n                else:\n                    preliminary = f\"FEASIBLE_{attempt['operation'].upper()}\"\n\n                receipt_data = {\n                    \"attempt_id\": att_id,\n                    \"status\": \"COMPLETED\",\n                    \"source_doc_identifier\": attempt[\"source_doc_identifier\"],\n                    \"source_image_id\": attempt[\"source_image_id\"],\n                    \"operation\": attempt[\"operation\"],\n                    \"original_text\": attempt[\"original_text\"],\n                    \"target_text\": attempt[\"target_text\"],\n                    \"inference_duration_sec\": inf_duration,\n                    \"peak_vram_mb\": torch.cuda.max_memory_allocated() / (1024**2) if has_cuda else 0.0,\n                    \"technical_qc\": {\n                        \"is_outside_bitwise_equal\": is_outside_equal,\n                        \"max_outside_diff\": max_outside_diff,\n                        \"inside_mean_l1\": in_mean_l1,\n                        \"changed_pixel_count\": changed_count,\n                        \"changed_pixel_ratio\": changed_ratio,\n                    },\n                    \"file_hashes\": {\n                        \"authentic_control_sha256\": compute_sha256(auth_path),\n                        \"generation_context_sha256\": compute_sha256(context_path),\n                        \"raw_model_output_sha256\": compute_sha256(raw_output_path),\n                        \"final_edited_document_sha256\": compute_sha256(edited_path),\n                        \"final_intervention_mask_sha256\": compute_sha256(mask_path),\n                    },\n                    \"preliminary_assessment\": preliminary,\n                }\n\n                with open(receipt_path, \"w\", encoding=\"utf-8\") as rf:\n                    json.dump(receipt_data, rf, indent=2)\n                attempt_records.append(receipt_data)\n\n            except Exception as exc:\n                print(f\"ERROR in {att_id}: {type(exc).__name__}: {exc}\")\n                failure_receipt = {\n                    \"attempt_id\": att_id,\n                    \"status\": \"FAILED\",\n                    \"operation\": attempt[\"operation\"],\n                    \"error_type\": type(exc).__name__,\n                    \"error_message\": str(exc),\n                    \"timestamp\": time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtime()),\n                }\n                fail_path = att_dir / \"attempt_failure_receipt.json\"\n                with open(fail_path, \"w\", encoding=\"utf-8\") as ff:\n                    json.dump(failure_receipt, ff, indent=2)\n                attempt_records.append(failure_receipt)\n\n        # 6. Stage 5: Human Review CSV, Contact Sheet, Manifest & Archive\n        csv_path = output_dir / \"human_review.csv\"\n        csv_fields = [\n            \"attempt_id\",\n            \"source_image_id\",\n            \"operation\",\n            \"original_text\",\n            \"target_text\",\n            \"outside_bitwise_equal\",\n            \"inside_mean_l1\",\n            \"changed_pixel_ratio\",\n            \"technical_verdict\",\n            \"human_verdict\",\n            \"human_notes\",\n        ]\n\n        csv_rows_to_write: List[Dict[str, Any]] = []\n        records_map = {r[\"attempt_id\"]: r for r in attempt_records}\n\n        for attempt in attempts:\n            att_id = attempt[\"attempt_id\"]\n            rec = records_map.get(att_id, {})\n            status = rec.get(\"status\", \"NOT_EXECUTED\")\n            tech_qc = rec.get(\"technical_qc\", {})\n\n            csv_rows_to_write.append(\n                {\n                    \"attempt_id\": att_id,\n                    \"source_image_id\": attempt[\"source_image_id\"],\n                    \"operation\": attempt[\"operation\"],\n                    \"original_text\": attempt[\"original_text\"],\n                    \"target_text\": attempt[\"target_text\"],\n                    \"outside_bitwise_equal\": tech_qc.get(\"is_outside_bitwise_equal\", \"N/A\"),\n                    \"inside_mean_l1\": f\"{tech_qc.get('inside_mean_l1', 0.0):.2f}\" if \"inside_mean_l1\" in tech_qc else \"N/A\",\n                    \"changed_pixel_ratio\": f\"{tech_qc.get('changed_pixel_ratio', 0.0):.4f}\" if \"changed_pixel_ratio\" in tech_qc else \"N/A\",\n                    \"technical_verdict\": rec.get(\"preliminary_assessment\", status),\n                    \"human_verdict\": \"PENDING_HUMAN_REVIEW\",\n                    \"human_notes\": \"\",\n                }\n            )\n\n        with open(csv_path, \"w\", newline=\"\", encoding=\"utf-8\") as cf:\n            writer = csv.DictWriter(cf, fieldnames=csv_fields, quoting=csv.QUOTE_MINIMAL)\n            writer.writeheader()\n            writer.writerows(csv_rows_to_write)\n        print(f\"\\nWritten human review CSV: {csv_path}\")\n\n        # Contact Sheet\n        try:\n            generate_contact_sheet(output_dir, attempts, attempt_records)\n        except Exception as cs_err:\n            print(f\"Warning: Contact sheet generation failed ({cs_err}). Continuing packaging.\")\n\n        # Final Manifest\n        manifest = {\n            \"run_id\": run_id,\n            \"plan_sha256\": plan_sha,\n            \"metadata\": meta,\n            \"preflight\": preflight,\n            \"checkpoint_budget\": checkpoint_receipt,\n            \"model_load_duration_sec\": load_duration,\n            \"attempts_summary\": attempt_records,\n            \"status\": \"COMPLETED_PENDING_HUMAN_QC\",\n        }\n        manifest_path = output_dir / \"feasibility_manifest.json\"\n        with open(manifest_path, \"w\", encoding=\"utf-8\") as mf:\n            json.dump(manifest, mf, indent=2)\n\n        # Zip Archive\n        zip_path = output_dir / \"document_feasibility_results.zip\"\n        print(f\"Packaging archive to {zip_path}...\")\n        with zipfile.ZipFile(zip_path, \"w\", zipfile.ZIP_DEFLATED) as zf:\n            for root, _, files in os.walk(output_dir):\n                for file in files:\n                    p = Path(root) / file\n                    if p == zip_path or p.suffix == \".part\" or \"cord_cache\" in p.parts:\n                        continue\n                    rel_p = p.relative_to(output_dir)\n                    zf.write(p, rel_p)\n\n        archive_size_bytes = zip_path.stat().st_size\n        archive_size_mb = archive_size_bytes / (1024**2)\n        print(f\"Archive created: {zip_path} ({archive_size_mb:.2f} MB)\")\n\n        if archive_size_bytes > budget[\"artifacts_transfer_max_bytes\"]:\n            raise ValueError(\n                f\"Archive size {archive_size_bytes} exceeds budget limit {budget['artifacts_transfer_max_bytes']} bytes!\"\n            )\n\n        print(\"=== Generation Feasibility Study Execution Complete ===\")\n        return {\n            \"status\": \"EXECUTED_PENDING_HUMAN_QC\",\n            \"run_id\": run_id,\n            \"archive_path\": str(zip_path),\n            \"archive_size_mb\": archive_size_mb,\n            \"attempts\": attempt_records,\n        }\n\n    except Exception as run_err:\n        print(f\"\\n[FATAL] Pipeline execution encountered error: {type(run_err).__name__}: {run_err}\")\n        run_failure_receipt = {\n            \"run_id\": run_id,\n            \"plan_sha256\": plan_sha,\n            \"error_type\": type(run_err).__name__,\n            \"error_message\": str(run_err),\n            \"timestamp\": time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtime()),\n        }\n        failure_path = output_dir / \"run_failure_receipt.json\"\n        with open(failure_path, \"w\", encoding=\"utf-8\") as ff:\n            json.dump(run_failure_receipt, ff, indent=2)\n\n        # Attempt to package partial zip archive if any artifacts exist\n        partial_zip = output_dir / \"document_feasibility_results_partial.zip\"\n        try:\n            with zipfile.ZipFile(partial_zip, \"w\", zipfile.ZIP_DEFLATED) as pzf:\n                for root, _, files in os.walk(output_dir):\n                    for file in files:\n                        p = Path(root) / file\n                        if p == partial_zip or p.suffix == \".part\" or \"cord_cache\" in p.parts:\n                            continue\n                        rel_p = p.relative_to(output_dir)\n                        pzf.write(p, rel_p)\n            print(f\"Partial results archive packaged to: {partial_zip}\")\n        except Exception:\n            pass\n\n        raise run_err\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=\"Run Document Feasibility Study on Colab GPU.\")\n    parser.add_argument(\n        \"--plan\",\n        type=Path,\n        default=Path(\"research/evidence/document_pilot/document_feasibility_plan.json\"),\n    )\n    parser.add_argument(\n        \"--output-dir\",\n        type=Path,\n        default=Path(\"research/evidence/document_pilot/feasibility_results\"),\n    )\n    parser.add_argument(\n        \"--dry-run\",\n        action=\"store_true\",\n        help=\"Perform preflight checks without downloading model or inferencing.\",\n    )\n    parser.add_argument(\n        \"--source-preflight-only\",\n        action=\"store_true\",\n        help=\"Perform source preflight and budget preflight only, then exit cleanly.\",\n    )\n    parser.add_argument(\"--device\", type=str, default=\"cuda\", choices=[\"cuda\", \"cpu\"])\n    args = parser.parse_args()\n\n    run_feasibility_pipeline(\n        plan_path=args.plan,\n        output_dir=args.output_dir,\n        dry_run=args.dry_run or args.source_preflight_only,\n        device=args.device,\n    )\n\n\nif __name__ == \"__main__\":\n    main()\n"

plan_file = Path('document_feasibility_plan.json')
with open(plan_file, 'w', encoding='utf-8', newline='\n') as f:
    f.write(PLAN_JSON)

runner_file = Path('run_document_feasibility_colab.py')
with open(runner_file, 'w', encoding='utf-8', newline='\n') as f:
    f.write(RUNNER_SOURCE)

def compute_sha(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while chunk := f.read(65536): h.update(chunk)
    return h.hexdigest()

actual_plan_sha = compute_sha(plan_file)
actual_runner_sha = compute_sha(runner_file)

assert actual_plan_sha == EXPECTED_PLAN_SHA, f'Plan SHA mismatch: {actual_plan_sha} != {EXPECTED_PLAN_SHA}'
assert actual_runner_sha == EXPECTED_RUNNER_SHA, f'Runner SHA mismatch: {actual_runner_sha} != {EXPECTED_RUNNER_SHA}'

plan_data = json.loads(PLAN_JSON)
print(f"[OK] Đã ghi và kiểm toán kế hoạch tiền đăng ký (SHA-256: {actual_plan_sha[:12]}...)")
print(f"  - Nghiên cứu: {plan_data['metadata']['study_name']}")
print(f"  - Protocol: v{plan_data['metadata']['protocol_version']}")
print(f"  - Model: {plan_data['metadata']['model']['repository']} @ {plan_data['metadata']['model']['revision'][:8]}")
print(f"  - Attempts: {len(plan_data['attempts'])} ({[a['attempt_id'] for a in plan_data['attempts']]})")
print(f"  - Seeds: {[a['seed'] for a in plan_data['attempts']]}")
print(f"[OK] Đã ghi và kiểm toán runner script (SHA-256: {actual_runner_sha[:12]}...)")


In [ ]:
# Cell 3: Thực thi chạy 4 attempts tạo dữ liệu khả thi
!python run_document_feasibility_colab.py --plan document_feasibility_plan.json --output-dir ./results


In [ ]:
# Cell 4: Kiểm tra kết quả và tải file archive về LOCAL
from pathlib import Path
try:
    from google.colab import files
    is_colab = True
except ImportError:
    is_colab = False

zip_path = Path('./results/document_feasibility_results.zip')
partial_zip = Path('./results/document_feasibility_results_partial.zip')
contact_sheet = Path('./results/feasibility_contact_sheet.png')
review_csv = Path('./results/human_review.csv')

target_zip = zip_path if zip_path.exists() else (partial_zip if partial_zip.exists() else None)
if target_zip:
    size_mb = target_zip.stat().st_size / (1024**2)
    print(f"[OK] Archive sẵn sàng: {target_zip.name} ({size_mb:.2f} MB)")
    if contact_sheet.exists():
        print(f"[OK] Contact sheet sẵn sàng: {contact_sheet.name}")
    if review_csv.exists():
        print(f"[OK] Human review CSV sẵn sàng: {review_csv.name}")
    if is_colab:
        print(f"Đang tải {target_zip.name} về máy LOCAL...")
        files.download(str(target_zip))
    else:
        print(f"Môi trường ngoài Colab: file lưu tại {target_zip.resolve()}")
    print("\nSau khi tải xong, hãy đặt file vào thư mục dự án và chạy:")
    print(f"  python scripts/research/verify_document_feasibility_artifacts.py --zip-path {target_zip.name}")
else:
    print("LỖI: Không tìm thấy file archive kết quả! Hãy kiểm tra log Cell 3.")
